# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.]` и `[0 1 1]`: `/` делит, а `//` округляет результат вниз.
2. `[4. 4. 4.]`: `axis=1` складывает четыре числа в каждой строке.
3. `[[0, 1, 2], [1, 2, 3], [2, 3, 4]]`: broadcasting складывает строку со столбцом.
4. `6.0`: корни равны 1, 2 и 3, их сумма равна 6.
5. `7.5`: среднее квадратов равно (1 + 4 + 9 + 16) / 4.


In [1]:
# Проверяем ответы после объяснений выше.
import numpy as np

a = np.array([2, 4, 6])
print(a / 2, a // 4)
m = np.ones((3, 4))
print(m.sum(axis=1))
print(np.arange(3) + np.arange(3).reshape(3, 1))
x = np.array([1, 4, 9])
print(np.sqrt(x).sum())
print((np.arange(1, 5) ** 2).mean())


[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# Преобразуем строки CSV в массивы для расчётов NumPy.
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [4]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
# return_index=True возвращает позиции первых появлений каждого номера заказа.
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [5]:
september = clean & (month == "2026-09")
october = clean & (month == "2026-10")
sep_revenue = amount[september].sum()
oct_revenue = amount[october].sum()
change = (oct_revenue - sep_revenue) / sep_revenue * 100

print("Чистых заказов:", clean.sum())
print(f"Сентябрь: {sep_revenue:,.0f}; октябрь: {oct_revenue:,.0f}")
print(f"Менеджеру: выручка изменилась на {change:.2f}%, что совпадает с очищенными данными ДЗ 2.")


Чистых заказов: 4739
Сентябрь: 35,243,410; октябрь: 34,419,520
Менеджеру: выручка изменилась на -2.34%, что совпадает с очищенными данными ДЗ 2.


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [6]:
categories = np.unique(cat)
revenue = np.zeros((len(categories), 2))

# Строки — категории, столбцы — сентябрь и октябрь.
for i, category in enumerate(categories):
    revenue[i, 0] = amount[september & (cat == category)].sum()
    revenue[i, 1] = amount[october & (cat == category)].sum()

print("Категория | Сентябрь | Октябрь")
for category, values in zip(categories, revenue):
    print(f"{category}: {values[0]:,.0f} | {values[1]:,.0f}")

total = revenue.sum(axis=1)  # Сумма двух месяцев для каждой категории.
category_change = (revenue[:, 1] - revenue[:, 0]) / revenue[:, 0] * 100
leader = total.argmax()
worst = category_change.argmin()
print(f"Менеджеру: лидер — {categories[leader]} ({total[leader]:,.0f}), самое сильное падение — {categories[worst]} ({category_change[worst]:.2f}%).")


Категория | Сентябрь | Октябрь
Дом: 4,997,730 | 5,125,790
Книги: 1,065,300 | 1,140,910
Красота: 2,610,950 | 2,678,990
Одежда: 9,167,100 | 8,505,730
Электроника: 17,402,330 | 16,968,100
Менеджеру: лидер — Электроника (34,370,430), самое сильное падение — Одежда (-7.21%).


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [7]:
counts = np.zeros((len(categories), 2), dtype=int)
for i, category in enumerate(categories):
    counts[i, 0] = (september & (cat == category)).sum()
    counts[i, 1] = (october & (cat == category)).sum()

# True считается как 1; деление таблиц даёт средний чек каждой группы.
average = revenue / counts
print("Заказы: сентябрь | октябрь; средний чек октября")
for i, category in enumerate(categories):
    print(f"{category}: {counts[i, 0]} | {counts[i, 1]}; {average[i, 1]:,.2f}")

best = average[:, 1].argmax()
print(f"Менеджеру: самый высокий средний чек октября у категории {categories[best]} — {average[best, 1]:,.2f}.")


Заказы: сентябрь | октябрь; средний чек октября
Дом: 476 | 476; 10,768.47
Книги: 265 | 299; 3,815.75
Красота: 422 | 435; 6,158.60
Одежда: 678 | 652; 13,045.60
Электроника: 531 | 505; 33,600.20
Менеджеру: самый высокий средний чек октября у категории Электроника — 33,600.20.


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [8]:
cities = np.unique(city)
table = np.zeros((len(cities), 2))
for i, name in enumerate(cities):
    table[i, 0] = amount[september & (city == name)].sum()
    table[i, 1] = amount[october & (city == name)].sum()

# axis=0 даёт итог каждого месяца; broadcasting делит на него весь столбец.
city_share = table / table.sum(axis=0)
print("Доли выручки: сентябрь | октябрь")
for name, values in zip(cities, city_share):
    print(f"{name}: {values[0]:.2%} | {values[1]:.2%}")
print("Суммы столбцов:", city_share.sum(axis=0))

# Маска выбирает строку Шымкента; [0] берёт её из таблицы с одной строкой.
shymkent_share = city_share[cities == "Шымкент"][0]
share_change = (shymkent_share[1] - shymkent_share[0]) * 100
print(f"Менеджеру: доля Шымкента изменилась на {share_change:.2f} процентного пункта.")


Доли выручки: сентябрь | октябрь
Актобе: 7.31% | 7.38%
Алматы: 41.15% | 42.54%
Астана: 25.63% | 25.71%
Атырау: 2.62% | 2.82%
Караганда: 10.35% | 9.80%
Шымкент: 12.93% | 11.75%
Суммы столбцов: [1. 1.]
Менеджеру: доля Шымкента изменилась на -1.19 процентного пункта.


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [9]:
city_change = table[:, 1] - table[:, 0]
contribution = city_change / city_change.sum() * 100

# Отрицательный вклад означает рост города, который компенсирует падение.
for name, value in zip(cities, contribution):
    print(f"{name}: {value:.2f}%")
print(f"Менеджеру: Шымкент даёт {contribution[cities == 'Шымкент'][0]:.2f}% общего падения; причины стоит искать в его заказах.")


Актобе: 4.44%
Алматы: -17.23%
Астана: 22.30%
Атырау: -5.57%
Караганда: 33.60%
Шымкент: 62.46%
Менеджеру: Шымкент даёт 62.46% общего падения; причины стоит искать в его заказах.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [10]:
# У table форма (6, 2), а у суммы строк — (6,): последние размеры не совпадают.
# keepdims=True сохраняет форму (6, 1), чтобы делить каждую строку на её итог.
share = table / table.sum(axis=1, keepdims=True)
for name, values in zip(cities, share):
    print(f"{name}: сентябрь {values[0]:.2%}, октябрь {values[1]:.2%}")
print("Суммы долей внутри городов:", share.sum(axis=1))
print("Менеджеру: доли двух месяцев внутри каждого города в сумме дают 100%.")


Актобе: сентябрь 50.36%, октябрь 49.64%
Алматы: сентябрь 49.76%, октябрь 50.24%
Астана: сентябрь 50.51%, октябрь 49.49%
Атырау: сентябрь 48.79%, октябрь 51.21%
Караганда: сентябрь 51.97%, октябрь 48.03%
Шымкент: сентябрь 52.99%, октябрь 47.01%
Суммы долей внутри городов: [1. 1. 1. 1. 1. 1.]
Менеджеру: доли двух месяцев внутри каждого города в сумме дают 100%.


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [11]:
x = amount[clean]
# z показывает, на сколько стандартных отклонений заказ отличается от среднего.
z = (x - x.mean()) / x.std()
unusual = np.abs(z) > 3
print(f"Среднее z: {z.mean():.6f}; std: {z.std():.6f}")
print("Среднее равно 0:", round(z.mean(), 6) == 0)
print("Стандартное отклонение равно 1:", round(z.std(), 6) == 1)
print(f"Менеджеру: {unusual.sum()} заказов ({unusual.mean():.2%}) имеют |z| > 3; их стоит проверить отдельно.")


Среднее z: 0.000000; std: 1.000000
Среднее равно 0: True
Стандартное отклонение равно 1: True
Менеджеру: 87 заказов (1.84%) имеют |z| > 3; их стоит проверить отдельно.


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [12]:
# NaN — пропуск; ~np.isnan(days) оставляет только известные сроки доставки.
valid = clean & ~np.isnan(days)
X = np.column_stack((amount[valid], days[valid]))
scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

print("Минимумы:", scaled.min(axis=0))
print("Максимумы:", scaled.max(axis=0))
print(f"Менеджеру: для {len(X)} заказов оба признака приведены к [0, 1], чтобы большие суммы не перевешивали сроки доставки только из-за масштаба.")


Минимумы: [0. 0.]
Максимумы: [1. 1.]
Менеджеру: для 4595 заказов оба признака приведены к [0, 1], чтобы большие суммы не перевешивали сроки доставки только из-за масштаба.
